# Spaceship Titanic | 01. Data Understanding & Exploratory Analysis

**Objective.** Examine the competition's labelled passenger data, understand the binary target `Transported`, investigate missingness, and explore relationships that may inform later classification experiments.

**Data.** The competition provides `train.csv` (labelled passengers), `test.csv` (unlabelled passengers), and `sample_submission.csv` (required prediction layout). Obtain these files from the Kaggle competition and place them in the project root. This notebook works whether it is opened from the root or a `notebooks/` subfolder.

**Scope.** This notebook is exploratory: it does **not** impute missing values, fit a model, or claim causal relationships. Model preprocessing and validation are handled in subsequent notebooks.

## 1. Load and inspect the competition data

In [1]:
# Import pandas for reading and exploring tabular datasets.
import pandas as pd

# Import Path to manage file locations.
from pathlib import Path

# Find the competition data whether the notebook is run from the
# project root or the project's notebooks/ directory.
working_dir = Path.cwd()
project_root = next(
    (folder for folder in [working_dir, working_dir.parent]
     if (folder / "train.csv").exists()
     and (folder / "test.csv").exists()
     and (folder / "sample_submission.csv").exists()),
    None
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find train.csv, test.csv, and sample_submission.csv "
        "in the current directory or its parent. Place the Kaggle files "
        "in the project root before running this notebook."
    )

# Define the paths to the three competition files.
train_path = project_root / "train.csv"
test_path = project_root / "test.csv"
submission_path = project_root / "sample_submission.csv"

# Load the training dataset.
train_df = pd.read_csv(train_path)

# Load the test dataset.
test_df = pd.read_csv(test_path)

# Load the sample submission file.
sample_submission = pd.read_csv(submission_path)

# Display the number of rows and columns in each dataset.
print("Training dataset:", train_df.shape)
print("Test dataset:", test_df.shape)
print("Sample submission:", sample_submission.shape)

Training dataset: (8693, 14)
Test dataset: (4277, 13)
Sample submission: (4277, 2)


In [2]:
# Display the first five passengers in the training dataset.
# This helps us understand how the data is organized.

display(train_df.head())

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
# Display all column names in the training dataset.
# We need to understand the available predictors and target variable.

print("TRAINING DATASET COLUMNS:\n")

for column in train_df.columns:
    print(column)

TRAINING DATASET COLUMNS:

PassengerId
HomePlanet
CryoSleep
Cabin
Destination
Age
VIP
RoomService
FoodCourt
ShoppingMall
Spa
VRDeck
Name
Transported


In [4]:
# Display a summary of the training dataset.
# This includes column names, non-missing observations,
# and the data type of each variable.

train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 891.5+ KB


## 2. Target distribution

`Transported` is the binary classification target. A class balance check helps establish a naïve majority-class benchmark and informs the choice of stratified validation.

In [5]:
# Count the number of passengers in each target category.
# True means transported; False means not transported.

train_df["Transported"].value_counts()

Transported
True     4378
False    4315
Name: count, dtype: int64

In [6]:
# Calculate the percentage of passengers in each target class.
# normalize=True converts frequencies into proportions.
# Multiplying by 100 converts proportions into percentages.

target_percentages = (
    train_df["Transported"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

# Display the percentages.
display(target_percentages)

Transported
True     50.36
False    49.64
Name: proportion, dtype: float64

## 3. Missing-value assessment

Count both incomplete **rows** and missing entries **by column**. Missingness should not automatically be treated as zero, particularly for passenger spending. We defer model-specific imputation to training pipelines to avoid leaking fitted imputation statistics into validation data.

In [7]:
# Count passengers with at least one missing value.
# isna() identifies missing values.
# any(axis=1) checks whether each row contains at least one missing value.

rows_with_missing = train_df.isna().any(axis=1).sum()

# Count passengers without any missing values.
complete_rows = train_df.notna().all(axis=1).sum()

# Calculate the percentage of passengers with missing information.
missing_percentage = (rows_with_missing / len(train_df)) * 100

# Display the results.
print("Total passengers:", len(train_df))
print("Passengers with missing values:", rows_with_missing)
print("Passengers with complete information:", complete_rows)
print("Percentage with missing values:", round(missing_percentage, 2), "%")

Total passengers: 8693
Passengers with missing values: 2087
Passengers with complete information: 6606
Percentage with missing values: 24.01 %


In [8]:
# Calculate the number of missing values in each column.
missing_count = train_df.isna().sum()

# Calculate the percentage of missing values in each column.
missing_percent = (
    train_df.isna().mean() * 100
)

# Combine the results into a summary table.
missing_summary = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percent": missing_percent
})

# Sort variables from highest to lowest missing percentage.
missing_summary = missing_summary.sort_values(
    by="Missing_Percent",
    ascending=False
)

# Display the summary table.
display(missing_summary.round(2))

,Missing_Count,Missing_Percent
CryoSleep,217,2.50
ShoppingMall,208,2.39
VIP,203,2.34
HomePlanet,201,2.31
Name,200,2.30
Cabin,199,2.29
VRDeck,188,2.16
Spa,183,2.11
FoodCourt,183,2.11
Destination,182,2.09


## 4. Passenger-group structure

The portion of `PassengerId` preceding `_` is used as a group identifier. First examine group sizes **within the labelled training data**; later notebooks also examine the combined train/test passenger manifest. Group membership does not imply identical outcomes.

In [9]:
# Create a working copy to preserve the original training data.
group_df = train_df.copy()

# Extract the group identifier from PassengerId.
# Example: "0003_02" becomes group "0003".
group_df["PassengerGroup"] = (
    group_df["PassengerId"]
    .str.split("_")
    .str[0]
)

# Display the first ten passenger IDs and their extracted groups.
display(
    group_df[["PassengerId", "PassengerGroup"]].head(10)
)

,PassengerId,PassengerGroup
0,0001_01,0001
1,0002_01,0002
2,0003_01,0003
3,0003_02,0003
4,0004_01,0004
5,0005_01,0005
6,0006_01,0006
7,0006_02,0006
8,0007_01,0007
9,0008_01,0008


In [10]:
# Count how many passengers belong to each passenger group.
group_sizes = (
    group_df.groupby("PassengerGroup")
    .size()
)

# Display descriptive statistics for group sizes.
print("PASSENGER GROUP SIZE SUMMARY")
display(group_sizes.describe())

# Count the number of groups containing more than one passenger.
multiple_passenger_groups = (group_sizes > 1).sum()

# Calculate the total number of unique passenger groups.
total_groups = group_sizes.size

print("\nTotal passenger groups:", total_groups)

print(
    "Groups containing more than one passenger:",
    multiple_passenger_groups
)

# Display the ten largest groups.
print("\nTEN LARGEST GROUPS:")
display(group_sizes.sort_values(ascending=False).head(10))

PASSENGER GROUP SIZE SUMMARY


count    6217.000000
mean        1.398263
std         0.944051
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         8.000000
dtype: float64


Total passenger groups: 6217
Groups containing more than one passenger: 1412

TEN LARGEST GROUPS:


PassengerGroup
9081    8
4005    8
8988    8
5133    8
4256    8
4498    8
8956    8
8796    8
0984    8
5885    8
dtype: int64

In [11]:
# Count the number of distinct transportation outcomes in each group.
# A value of 1 means every passenger in that group has the same outcome.
# A value of 2 means the group contains both True and False outcomes.

group_outcomes = (
    group_df
    .groupby("PassengerGroup")["Transported"]
    .nunique()
)

# Identify groups containing more than one passenger.
multi_group_ids = group_sizes[
    group_sizes > 1
].index

# Select outcome information for multi-passenger groups only.
multi_group_outcomes = group_outcomes.loc[
    multi_group_ids
]

# Count groups where all passengers share the same outcome.
same_outcome_groups = (
    multi_group_outcomes == 1
).sum()

# Count groups containing both outcomes.
mixed_outcome_groups = (
    multi_group_outcomes == 2
).sum()

# Calculate the proportion of multi-passenger groups
# where all passengers share the same transportation outcome.
same_outcome_percent = (
    same_outcome_groups / len(multi_group_outcomes)
) * 100

print("GROUP OUTCOME ANALYSIS")
print("-" * 40)

print("Multi-passenger groups:", len(multi_group_outcomes))
print("Groups with identical outcomes:", same_outcome_groups)
print("Groups with mixed outcomes:", mixed_outcome_groups)

print(
    "Percentage with identical outcomes:",
    round(same_outcome_percent, 2),
    "%"
)

GROUP OUTCOME ANALYSIS
----------------------------------------
Multi-passenger groups: 1412
Groups with identical outcomes: 615
Groups with mixed outcomes: 797
Percentage with identical outcomes: 43.56 %


In [12]:
# Identify group IDs containing both transported and
# non-transported passengers.

mixed_group_ids = multi_group_outcomes[
    multi_group_outcomes == 2
].index

# Select a small number of mixed-outcome groups for inspection.
example_groups = mixed_group_ids[:5]

# Display the passengers belonging to these groups.
display(
    group_df.loc[
        group_df["PassengerGroup"].isin(example_groups),
        [
            "PassengerId",
            "PassengerGroup",
            "HomePlanet",
            "CryoSleep",
            "Cabin",
            "Transported"
        ]
    ].sort_values("PassengerGroup")
)

,PassengerId,PassengerGroup,HomePlanet,CryoSleep,Cabin,Transported
19,0017_01,0017,Earth,False,G/0/P,True
20,0017_02,0017,Earth,False,F/6/P,False
21,0020_01,0020,Earth,True,E/0/S,False
22,0020_02,0020,Earth,True,E/0/S,False
23,0020_03,0020,Earth,True,E/0/S,False
24,0020_04,0020,Earth,False,E/0/S,True
25,0020_05,0020,Earth,True,E/0/S,False
26,0020_06,0020,Earth,False,E/0/S,False
43,0044_01,0044,Earth,True,G/3/P,False
44,0044_02,0044,Earth,True,G/3/P,True


## 5. HomePlanet and transportation

Compare group counts and observed transportation proportions. Differences describe **associations in the labelled sample**, not causal effects.

In [13]:
# Count passengers from each home planet.
# dropna=False includes passengers whose home planet is missing.

homeplanet_counts = train_df["HomePlanet"].value_counts(
    dropna=False
)

# Display the number of passengers by home planet.
display(homeplanet_counts)

HomePlanet
Earth     4602
Europa    2131
Mars      1759
NaN        201
Name: count, dtype: int64

In [14]:
# Group passengers according to their home planet.
# The target is Boolean: True = 1 and False = 0.
# Therefore, the mean gives the proportion transported.

homeplanet_transport = (
    train_df
    .groupby("HomePlanet", dropna=False)["Transported"]
    .agg(
        Total_Passengers="count",
        Transported_Count="sum",
        Transported_Rate="mean"
    )
)

# Convert the transportation rate into a percentage.
homeplanet_transport["Transported_Percent"] = (
    homeplanet_transport["Transported_Rate"] * 100
)

# Display the results, rounded to two decimal places.
display(homeplanet_transport.round(2))

,Total_Passengers,Transported_Count,Transported_Rate,Transported_Percent
HomePlanet,,,,
Earth,4602,1951,0.42,42.39
Europa,2131,1404,0.66,65.88
Mars,1759,920,0.52,52.30
NaN,201,103,0.51,51.24


## 6. CryoSleep and its interaction with HomePlanet

Investigate whether the observed transportation proportion differs with `CryoSleep`, then cross-tabulate CryoSleep by HomePlanet. Missing categories are retained for transparency.

In [15]:
# Count passengers by CryoSleep status.
# dropna=False ensures missing values are included.

cryosleep_counts = train_df["CryoSleep"].value_counts(
    dropna=False
)

# Display the frequency distribution.
display(cryosleep_counts)

CryoSleep
False    5439
True     3037
NaN       217
Name: count, dtype: int64

In [16]:
# Group passengers according to their CryoSleep status.
# Include missing categories in the analysis.

cryosleep_transport = (
    train_df
    .groupby("CryoSleep", dropna=False)["Transported"]
    .agg(
        Total_Passengers="count",
        Transported_Count="sum",
        Transported_Rate="mean"
    )
)

# Convert the transportation rate into a percentage.
cryosleep_transport["Transported_Percent"] = (
    cryosleep_transport["Transported_Rate"] * 100
)

# Display the results rounded to two decimal places.
display(cryosleep_transport.round(2))

,Total_Passengers,Transported_Count,Transported_Rate,Transported_Percent
CryoSleep,,,,
False,5439,1789,0.33,32.89
True,3037,2483,0.82,81.76
NaN,217,106,0.49,48.85


In [17]:
# Create a frequency table showing HomePlanet and CryoSleep.
# dropna=False includes passengers with missing information.

homeplanet_cryosleep_counts = pd.crosstab(
    train_df["HomePlanet"].fillna("Missing"),
    train_df["CryoSleep"].astype("string").fillna("Missing")
)

# Display the frequency table.
display(homeplanet_cryosleep_counts)

CryoSleep,False,Missing,True
HomePlanet,,,
Earth,3106,114,1382
Europa,1162,58,911
Mars,1047,43,669
Missing,124,2,75


## 7. Onboard expenditure

Explore spending levels and their relationships with transportation. Many observed expenditures are zero, but **missing expenditure is not the same as zero expenditure**.

The early spending-status exploration is refined below after accounting for partially observed spending records.

In [18]:
# Create a copy to avoid modifying our original dataset.
eda_df = train_df.copy()

# Replace missing HomePlanet values with an explicit category.
eda_df["HomePlanet"] = (
    eda_df["HomePlanet"]
    .fillna("Missing")
)

# Convert CryoSleep to string so that missing values
# can be represented as their own category.
eda_df["CryoSleep"] = (
    eda_df["CryoSleep"]
    .astype("string")
    .fillna("Missing")
)

# Group passengers by HomePlanet and CryoSleep status.
# Calculate the number of passengers and transportation rate.
homeplanet_cryo_transport = (
    eda_df
    .groupby(["HomePlanet", "CryoSleep"])["Transported"]
    .agg(
        Total_Passengers="count",
        Transported_Count="sum",
        Transported_Rate="mean"
    )
)

# Convert the transportation rate to a percentage.
homeplanet_cryo_transport["Transported_Percent"] = (
    homeplanet_cryo_transport["Transported_Rate"] * 100
)

# Display the table.
display(homeplanet_cryo_transport.round(2))

Total_Passengers  Transported_Count  Transported_Rate  \
HomePlanet CryoSleep                                                          
Earth      False                  3106                997              0.32   
           Missing                 114                 47              0.41   
           True                   1382                907              0.66   
Europa     False                  1162                465              0.40   
           Missing                  58                 38              0.66   
           True                    911                901              0.99   
Mars       False                  1047                290              0.28   
           Missing                  43                 20              0.47   
           True                    669                610              0.91   
Missing    False                   124                 37              0.30   
           Missing                   2                  1              0.50   
           True                     75                 65              0.87   

                      Transported_Percent  
HomePlanet CryoSleep                       
Earth      False                    32.10  
           Missing                  41.23  
           True                     65.63  
Europa     False                    40.02  
           Missing                  65.52  
           True                     98.90  
Mars       False                    27.70  
           Missing                  46.51  
           True                     91.18  
Missing    False                    29.84  
           Missing                  50.00  
           True                     86.67

In [19]:
# Identify the five onboard spending variables.
spending_columns = [
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck"
]

# Calculate descriptive statistics for each spending variable.
# The transpose (.T) displays variables as rows for easier reading.
spending_summary = train_df[spending_columns].describe().T

# Display the summary table rounded to two decimal places.
display(spending_summary.round(2))

,count,mean,std,min,25%,50%,75%,max
RoomService,8512.0,224.69,666.72,0.0,0.0,0.0,47.0,14327.0
FoodCourt,8510.0,458.08,1611.49,0.0,0.0,0.0,76.0,29813.0
ShoppingMall,8485.0,173.73,604.70,0.0,0.0,0.0,27.0,23492.0
Spa,8510.0,311.14,1136.71,0.0,0.0,0.0,59.0,22408.0
VRDeck,8505.0,304.85,1145.72,0.0,0.0,0.0,46.0,24133.0


In [20]:
# Group passengers by their transportation outcome.
# Calculate the median for each onboard spending variable.
spending_by_transport = (
    train_df
    .groupby("Transported")[spending_columns]
    .median()
)

# Display median spending by transportation outcome.
display(spending_by_transport.round(2))

,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
Transported,,,,,
False,1.0,0.0,0.0,4.0,1.0
True,0.0,0.0,0.0,0.0,0.0


### 7.1 Initial spending-status exploration

This exploratory classification is an intermediate calculation. **Use the refined spending-status categories in Section 7.2** for conclusions and later feature engineering.

In [21]:
# Create a copy of the original dataset.
# This ensures that our original training data remains unchanged.
spending_df = train_df.copy()

# Identify the five onboard spending variables.
spending_columns = [
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck"
]

# Identify passengers with complete spending information.
# True means all five expenditure values are available.
spending_df["SpendingComplete"] = (
    spending_df[spending_columns].notna().all(axis=1)
)

# Calculate total spending only when all five values are known.
# min_count=5 prevents pandas from treating incomplete records
# as having a fully observed total.
spending_df["TotalSpending"] = (
    spending_df[spending_columns].sum(axis=1, min_count=5)
)

# Create a spending category.
# Preserve missing totals rather than calling them zero spending.
spending_df["SpendingCategory"] = "Unknown"

# Identify passengers who spent exactly zero.
spending_df.loc[
    spending_df["TotalSpending"] == 0,
    "SpendingCategory"
] = "Zero Spending"

# Identify passengers who spent more than zero.
spending_df.loc[
    spending_df["TotalSpending"] > 0,
    "SpendingCategory"
] = "Positive Spending"

# Display the number of passengers in each spending category.
display(
    spending_df["SpendingCategory"].value_counts()
)

SpendingCategory
Positive Spending    4538
Zero Spending        3247
Unknown               908
Name: count, dtype: int64

In [22]:
# Group passengers according to their spending category.
# Calculate the number of passengers and transportation rate.
spending_transport = (
    spending_df
    .groupby("SpendingCategory")["Transported"]
    .agg(
        Total_Passengers="count",
        Transported_Count="sum",
        Transported_Rate="mean"
    )
)

# Convert the transportation rate to a percentage.
spending_transport["Transported_Percent"] = (
    spending_transport["Transported_Rate"] * 100
)

# Display the results.
display(spending_transport.round(2))

,Total_Passengers,Transported_Count,Transported_Rate,Transported_Percent
SpendingCategory,,,,
Positive Spending,4538,1367,0.30,30.12
Unknown,908,466,0.51,51.32
Zero Spending,3247,2545,0.78,78.38


In [23]:
# Create a cross-tabulation of CryoSleep and spending categories.
# Convert CryoSleep to string to retain missing values.
cryo_spending_table = pd.crosstab(
    spending_df["CryoSleep"].astype("string").fillna("Missing"),
    spending_df["SpendingCategory"]
)

# Display the frequency table.
display(cryo_spending_table)

SpendingCategory,Positive Spending,Unknown,Zero Spending
CryoSleep,,,
False,4432,537,470
Missing,106,24,87
True,0,347,2690


### 7.2 Refined spending-status definition

- **Positive Spending:** at least one observed spending field is greater than zero (even if another field is missing).
- **Zero Spending:** all five fields are observed and all equal zero.
- **Unknown:** incomplete observations with no recorded positive spending.

This distinction prevents incomplete spending records from being silently labelled as zero spending.

In [24]:
# Create a new working copy of the original dataset.
# We preserve train_df so that our original data remains unchanged.
spending_refined = train_df.copy()

# Identify the five expenditure variables.
spending_columns = [
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck"
]

# Identify passengers with at least one recorded positive spending value.
# This confirms positive spending even if other values are missing.
has_positive_spending = (
    spending_refined[spending_columns].gt(0).any(axis=1)
)

# Identify passengers whose five spending values are all recorded.
all_spending_recorded = (
    spending_refined[spending_columns].notna().all(axis=1)
)

# Identify passengers whose five spending values are all zero.
confirmed_zero_spending = (
    spending_refined[spending_columns].eq(0).all(axis=1)
)

# Start by classifying all passengers as Unknown.
spending_refined["SpendingStatus"] = "Unknown"

# Assign passengers with confirmed positive spending.
spending_refined.loc[
    has_positive_spending,
    "SpendingStatus"
] = "Positive Spending"

# Assign passengers whose five spending values are all zero.
spending_refined.loc[
    all_spending_recorded & confirmed_zero_spending,
    "SpendingStatus"
] = "Zero Spending"

# Display the improved spending-category counts.
display(
    spending_refined["SpendingStatus"].value_counts()
)

SpendingStatus
Positive Spending    5040
Zero Spending        3247
Unknown               406
Name: count, dtype: int64

In [25]:
# Calculate transportation outcomes for the improved spending categories.
refined_spending_transport = (
    spending_refined
    .groupby("SpendingStatus")["Transported"]
    .agg(
        Total_Passengers="count",
        Transported_Count="sum",
        Transported_Rate="mean"
    )
)

# Convert transportation rates into percentages.
refined_spending_transport["Transported_Percent"] = (
    refined_spending_transport["Transported_Rate"] * 100
)

# Display the results.
display(refined_spending_transport.round(2))

,Total_Passengers,Transported_Count,Transported_Rate,Transported_Percent
SpendingStatus,,,,
Positive Spending,5040,1505,0.30,29.86
Unknown,406,328,0.81,80.79
Zero Spending,3247,2545,0.78,78.38


In [26]:
# Create a frequency table comparing CryoSleep and
# the improved spending categories.
refined_cryo_spending = pd.crosstab(
    spending_refined["CryoSleep"]
    .astype("string")
    .fillna("Missing"),

    spending_refined["SpendingStatus"]
)

# Display the cross-tabulation.
display(refined_cryo_spending)

SpendingStatus,Positive Spending,Unknown,Zero Spending
CryoSleep,,,
False,4921,48,470
Missing,119,11,87
True,0,347,2690


## 8. Age and transportation

Describe age by the observed class, then examine exploratory age bands and their association with CryoSleep. Age groups are descriptive choices rather than established natural thresholds.

In [27]:
# Group passengers according to their transportation outcome.
# Calculate descriptive statistics for passenger age.
# This helps us compare age distributions between the two classes.

age_by_transport = (
    train_df
    .groupby("Transported")["Age"]
    .describe()
)

# Display the results rounded to two decimal places.
display(age_by_transport.round(2))

,count,mean,std,min,25%,50%,75%,max
Transported,,,,,,,,
False,4226.0,29.92,13.70,0.0,21.0,27.0,38.0,79.0
True,4288.0,27.75,15.15,0.0,18.0,26.0,37.0,78.0


In [28]:
# Create a copy of the original dataset.
# This avoids modifying the original training data.
age_df = train_df.copy()

# Divide passenger age into four exploratory age groups.
# Missing ages remain missing.
age_df["AgeGroup"] = pd.cut(
    age_df["Age"],
    bins=[-1, 17, 35, 60, float("inf")],
    labels=[
        "Children",
        "Young Adults",
        "Middle-Aged Adults",
        "Older Adults"
    ]
)

# Calculate transportation rates by age group.
age_group_transport = (
    age_df
    .groupby("AgeGroup", observed=True)["Transported"]
    .agg(
        Total_Passengers="count",
        Transported_Count="sum",
        Transported_Rate="mean"
    )
)

# Convert transportation rates to percentages.
age_group_transport["Transported_Percent"] = (
    age_group_transport["Transported_Rate"] * 100
)

# Display the results.
display(age_group_transport.round(2))

,Total_Passengers,Transported_Count,Transported_Rate,Transported_Percent
AgeGroup,,,,
Children,1545,973,0.63,62.98
Young Adults,4456,2094,0.47,46.99
Middle-Aged Adults,2293,1117,0.49,48.71
Older Adults,220,104,0.47,47.27


In [29]:
# Create a frequency table showing the number of passengers
# in each age group according to their CryoSleep status.

age_cryosleep_counts = pd.crosstab(
    age_df["AgeGroup"],
    age_df["CryoSleep"].astype("string").fillna("Missing")
)

# Display the frequency table.
display(age_cryosleep_counts)

CryoSleep,False,Missing,True
AgeGroup,,,
Children,767,45,733
Young Adults,2927,112,1417
Middle-Aged Adults,1505,54,734
Older Adults,144,5,71


In [30]:
# Calculate row percentages to compare CryoSleep distribution
# across age groups of different sizes.

age_cryosleep_percent = pd.crosstab(
    age_df["AgeGroup"],
    age_df["CryoSleep"].astype("string").fillna("Missing"),
    normalize="index"
) * 100

# Display percentages rounded to two decimal places.
display(age_cryosleep_percent.round(2))

CryoSleep,False,Missing,True
AgeGroup,,,
Children,49.64,2.91,47.44
Young Adults,65.69,2.51,31.80
Middle-Aged Adults,65.63,2.35,32.01
Older Adults,65.45,2.27,32.27


In [31]:
# Create a copy of our age-group dataset.
age_cryo_df = age_df.copy()

# Convert CryoSleep into a categorical string variable.
# Preserve missing values as a separate category.
age_cryo_df["CryoSleep"] = (
    age_cryo_df["CryoSleep"]
    .astype("string")
    .fillna("Missing")
)

# Calculate transportation rates for each
# AgeGroup and CryoSleep combination.
age_cryo_transport = (
    age_cryo_df
    .groupby(
        ["AgeGroup", "CryoSleep"],
        observed=True
    )["Transported"]
    .agg(
        Total_Passengers="count",
        Transported_Count="sum",
        Transported_Rate="mean"
    )
)

# Convert proportions into percentages.
age_cryo_transport["Transported_Percent"] = (
    age_cryo_transport["Transported_Rate"] * 100
)

# Display the results.
display(age_cryo_transport.round(2))

Total_Passengers  Transported_Count  \
AgeGroup           CryoSleep                                        
Children           False                   767                369   
                   Missing                  45                 29   
                   True                    733                575   
Young Adults       False                  2927                883   
                   Missing                 112                 49   
                   True                   1417               1162   
Middle-Aged Adults False                  1505                472   
                   Missing                  54                 26   
                   True                    734                619   
Older Adults       False                   144                 39   
                   Missing                   5                  2   
                   True                     71                 63   

                              Transported_Rate  Transported_Percent  
AgeGroup           CryoSleep                                         
Children           False                  0.48                48.11  
                   Missing                0.64                64.44  
                   True                   0.78                78.44  
Young Adults       False                  0.30                30.17  
                   Missing                0.44                43.75  
                   True                   0.82                82.00  
Middle-Aged Adults False                  0.31                31.36  
                   Missing                0.48                48.15  
                   True                   0.84                84.33  
Older Adults       False                  0.27                27.08  
                   Missing                0.40                40.00  
                   True                   0.89                88.73

## 9. Cabin components

Split the structured `Cabin` field into `CabinDeck`, numeric `CabinNumber`, and `CabinSide`. These derived predictors are used consistently in the later modelling notebooks.

In [32]:
# Create a copy of the original training dataset.
# This protects our original data from accidental modification.
cabin_df = train_df.copy()

# Split the Cabin column into three separate components.
# Example: "B/0/P" becomes ["B", "0", "P"].
cabin_parts = cabin_df["Cabin"].str.split("/", expand=True)

# Extract the deck, cabin number, and side.
cabin_df["CabinDeck"] = cabin_parts[0]
cabin_df["CabinNumber"] = pd.to_numeric(
    cabin_parts[1],
    errors="coerce"
)
cabin_df["CabinSide"] = cabin_parts[2]

# Display the first ten passengers with their extracted cabin details.
display(
    cabin_df[
        ["Cabin", "CabinDeck", "CabinNumber", "CabinSide"]
    ].head(10)
)

,Cabin,CabinDeck,CabinNumber,CabinSide
0,B/0/P,B,0.0,P
1,F/0/S,F,0.0,S
2,A/0/S,A,0.0,S
3,A/0/S,A,0.0,S
4,F/1/S,F,1.0,S
5,F/0/P,F,0.0,P
6,F/2/S,F,2.0,S
7,G/0/S,G,0.0,S
8,F/3/S,F,3.0,S
9,B/1/P,B,1.0,P


In [33]:
# Calculate transportation rates for each cabin deck.
# Include missing cabin decks as a separate category.
deck_transport = (
    cabin_df
    .groupby("CabinDeck", dropna=False)["Transported"]
    .agg(
        Total_Passengers="count",
        Transported_Count="sum",
        Transported_Rate="mean"
    )
)

# Convert the transportation rate into a percentage.
deck_transport["Transported_Percent"] = (
    deck_transport["Transported_Rate"] * 100
)

# Display the summary.
display(
    deck_transport.round(2).sort_values(
        "Total_Passengers",
        ascending=False
    )
)

,Total_Passengers,Transported_Count,Transported_Rate,Transported_Percent
CabinDeck,,,,
F,2794,1229,0.44,43.99
G,2559,1321,0.52,51.62
E,876,313,0.36,35.73
B,779,572,0.73,73.43
C,747,508,0.68,68.01
D,478,207,0.43,43.31
A,256,127,0.50,49.61
NaN,199,100,0.50,50.25
T,5,1,0.20,20.00


In [34]:
# Calculate transportation rates by cabin side.
# Missing cabin sides are included in the analysis.
side_transport = (
    cabin_df
    .groupby("CabinSide", dropna=False)["Transported"]
    .agg(
        Total_Passengers="count",
        Transported_Count="sum",
        Transported_Rate="mean"
    )
)

# Convert the transportation rate into a percentage.
side_transport["Transported_Percent"] = (
    side_transport["Transported_Rate"] * 100
)

# Display the results.
display(side_transport.round(2))

,Total_Passengers,Transported_Count,Transported_Rate,Transported_Percent
CabinSide,,,,
P,4206,1898,0.45,45.13
S,4288,2380,0.56,55.50
NaN,199,100,0.50,50.25


## 10. Destination and transportation

Compare the observed class proportions across destinations, retaining missing destinations as a separate summary group.

In [35]:
# Calculate transportation rates for each destination.
# Include missing destination values.
destination_transport = (
    train_df
    .groupby("Destination", dropna=False)["Transported"]
    .agg(
        Total_Passengers="count",
        Transported_Count="sum",
        Transported_Rate="mean"
    )
)

# Convert transportation rates into percentages.
destination_transport["Transported_Percent"] = (
    destination_transport["Transported_Rate"] * 100
)

# Display the results.
display(destination_transport.round(2))

,Total_Passengers,Transported_Count,Transported_Rate,Transported_Percent
Destination,,,,
55 Cancri e,1800,1098,0.61,61.00
PSO J318.5-22,796,401,0.50,50.38
TRAPPIST-1e,5915,2787,0.47,47.12
NaN,182,92,0.51,50.55


## 11. Key EDA findings and modelling implications

| Topic | Observed result | Implication |
|---|---|---|
| Target balance | 4,378 transported (50.36%) vs 4,315 not transported (49.64%) | Stratified splitting is reasonable; accuracy is informative but should not be the only diagnostic. |
| Missingness | 2,087 of 8,693 rows (24.01%) contain at least one missing value | Avoid dropping all incomplete rows; compare model-appropriate missing-value treatments. |
| Group structure | 6,217 groups in the **labelled** dataset, including 1,412 multi-passenger groups | Consider group-aware validation as a robustness check. |
| HomePlanet | Observed transportation rates differ by planet | Include HomePlanet among candidate categorical features. |
| CryoSleep | Transportation rates differ strongly by status | Keep CryoSleep; examine possible interactions without making causal claims. |
| Spending | Refined categories distinguish 5,040 positive, 3,247 zero, and 406 unknown cases | Preserve missingness; evaluate engineered features rather than assuming they help. |
| Cabin | Deck, number, and side can be extracted from a structured string | Test cabin components as model predictors. |

**Limitations.** This is descriptive analysis of the labelled competition data. The data were examined before subsequent model selection, so later cross-validation results should be described as **development estimates**, not completely untouched test performance. Differences in observed transportation rates are associative and may reflect other features.

**Next notebook:** `02_model_development.ipynb` evaluates preprocessing choices and classifiers with training-only fitted transformations and validation procedures.